# Minimal Structural Analysis: Toy Connectome as Controller Prior

**Purpose — Project A scaffold**  
This notebook demonstrates the graph analysis pipeline for the FlyWire Summer Internship application.  
The goal is to extract structural metrics from a connectome-like directed graph and interpret them as  
**inductive biases** a connectome-shaped GNN controller would inherit.

**Data**: `../data/toy_connectome_nodes.csv` and `../data/toy_connectome_edges.csv`  
30-node synthetic graph with `sensor / inter / motor` layers, dense inter-layer recurrence,  
and sparse feedback. Replace with the real FlyWire v630 snapshot from  
[Zenodo:10676866](https://zenodo.org/records/10676866) to run Project A on real data.

**Analysis pipeline**:
1. Load & visualize the graph
2. Degree distribution by layer
3. Betweenness centrality — bottleneck detection
4. Feedforward / feedback / lateral weight balance
5. Hub detection (rich-club in inter-layer)
6. Global hierarchy score
7. Controller-prior summary table

In [ ]:
import pandas as pd
import networkx as nx
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from collections import defaultdict

plt.rcParams.update({'figure.dpi': 120, 'font.size': 9})

LAYER_COLOR = {'sensor': '#4c9be8', 'inter': '#f5a623', 'motor': '#4caf50'}
LAYER_RANK  = {'sensor': 0, 'inter': 1, 'motor': 2}
layer_patches = [mpatches.Patch(color=c, label=l) for l, c in LAYER_COLOR.items()]

## 0. Load Data & Visualise

In [ ]:
nodes_df = pd.read_csv('../data/toy_connectome_nodes.csv')
edges_df = pd.read_csv('../data/toy_connectome_edges.csv')

print(f"Nodes: {len(nodes_df)}   Edges: {len(edges_df)}")
print(nodes_df['layer'].value_counts().to_string())
nodes_df.head()

In [ ]:
G = nx.DiGraph()
for _, row in nodes_df.iterrows():
    G.add_node(row['node'], layer=row['layer'])
for _, row in edges_df.iterrows():
    G.add_edge(row['src'], row['dst'], weight=float(row['weight']))

node_list   = list(G.nodes)
node_colors = [LAYER_COLOR[G.nodes[n]['layer']] for n in node_list]

pos = nx.spring_layout(G, seed=42, k=1.8)
fig, ax = plt.subplots(figsize=(11, 7))
nx.draw_networkx_nodes(G, pos, nodelist=node_list, node_color=node_colors,
                       node_size=320, ax=ax, alpha=0.92)
nx.draw_networkx_edges(G, pos, ax=ax, arrowsize=10, alpha=0.30, edge_color='#555')
nx.draw_networkx_labels(G, pos, ax=ax, font_size=7)
ax.legend(handles=layer_patches, loc='upper left', fontsize=9)
ax.set_title('Toy connectome — sensor (blue) → inter (orange) → motor (green)')
ax.axis('off')
plt.tight_layout()
plt.show()

## 1. Degree Distribution by Layer

**In-degree**: how many presynaptic partners a neuron integrates from.  
**Out-degree**: fan-out to downstream targets.

**Controller prior**:
- *Sensor* nodes: low in-degree (driven by environment), moderate-high out-degree (fan-out to inter).
- *Motor* nodes: moderate in-degree (converging inter signals), low out-degree (actuator outputs).
- *Inter* neurons: widest range — hubs emerge here and determine message-passing capacity.

In a GNN controller these degree statistics set how much information each node aggregates per step.  
The real FlyWire connectome has ~30% rich-club inter-neurons (Lin et al. 2024, *Nature*).

In [ ]:
in_deg  = dict(G.in_degree(weight='weight'))
out_deg = dict(G.out_degree(weight='weight'))

nodes_df = nodes_df.copy()
nodes_df['in_degree']  = nodes_df['node'].map(in_deg)
nodes_df['out_degree'] = nodes_df['node'].map(out_deg)
nodes_df['total_deg']  = nodes_df['in_degree'] + nodes_df['out_degree']

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, col in zip(axes, ['in_degree', 'out_degree']):
    for layer, grp in nodes_df.groupby('layer'):
        ax.scatter([layer] * len(grp), grp[col], c=LAYER_COLOR[layer],
                   s=80, alpha=0.85, label=layer, zorder=3)
    ax.set_xlabel('Layer')
    ax.set_ylabel(col.replace('_', ' ').title())
    ax.set_title(col.replace('_', ' ').title() + ' by Layer')
    ax.legend()
plt.tight_layout()
plt.show()

print(nodes_df.groupby('layer')[['in_degree', 'out_degree', 'total_deg']].describe().round(2))

## 2. Betweenness Centrality — Bottleneck Detection

Betweenness centrality counts how often a node lies on the **shortest path** between other node pairs.  
High-betweenness neurons are **information bottlenecks** — disrupting them fragments information flow.

**Controller prior**: High-betweenness inter-neurons are natural **sub-policy hubs**.  
In a hierarchical controller they mediate computations shared across multiple sensorimotor pathways.  
In GNN terms, their hidden states carry the most *mixed* information after message passing —  
a signal to allocate higher embedding dimensionality or wider MLP readouts to these nodes.

In [ ]:
bc = nx.betweenness_centrality(G, normalized=True, weight='weight')
nodes_df['betweenness'] = nodes_df['node'].map(bc)

bar_colors = [LAYER_COLOR[G.nodes[n]['layer']] for n in nodes_df['node']]

fig, ax = plt.subplots(figsize=(13, 4))
ax.bar(nodes_df['node'], nodes_df['betweenness'], color=bar_colors, edgecolor='white')
ax.set_xlabel('Node')
ax.set_ylabel('Betweenness centrality (normalised)')
ax.set_title('Betweenness centrality per neuron')
ax.legend(handles=layer_patches)
plt.xticks(rotation=90, fontsize=7)
plt.tight_layout()
plt.show()

print("Top-8 bottleneck neurons:")
print(nodes_df.nlargest(8, 'betweenness')[['node', 'layer', 'betweenness']].to_string(index=False))

## 3. Feedforward / Feedback / Lateral Balance

We classify every directed edge by its layer-to-layer direction:
- **Feedforward (FF)**: sensor→inter or inter→motor — drives action.
- **Feedback (FB)**: motor→inter or inter→sensor — enables correction, state estimation.
- **Lateral**: within-layer recurrence in inter-layer — creates short-term memory capacity.

**Controller prior**:
| Fraction | High value means… | GNN implementation |
|----------|-------------------|--------------------|
| FF       | Rapid reactive throughput | Forward message passes dominate |
| FB       | Online correction, efference copy | Residual skip-connections / self-loops |
| Lateral  | Working-memory / temporal integration | Keep recurrent edges in adjacency matrix |

Lin et al. 2024 found the real Drosophila connectome sits at an intermediate FB/lateral balance —  
reactive *and* integrative, consistent with robust locomotion under perturbation.

In [ ]:
stats = defaultdict(float)
for u, v, d in G.edges(data=True):
    w   = d.get('weight', 1.0)
    r_u = LAYER_RANK[G.nodes[u]['layer']]
    r_v = LAYER_RANK[G.nodes[v]['layer']]
    if   r_v > r_u: stats['feedforward'] += w
    elif r_v < r_u: stats['feedback']    += w
    else:           stats['lateral']     += w

total  = sum(stats.values())
labels = ['feedforward', 'feedback', 'lateral']
values = [stats[k] for k in labels]
pcts   = [100 * v / total for v in values]

for l, v, p in zip(labels, values, pcts):
    print(f"  {l:<14}: {v:6.1f}  ({p:.1f}%)")

fig, ax = plt.subplots(figsize=(5, 4))
bars = ax.bar(labels, values, color=['cornflowerblue', 'tomato', 'goldenrod'])
ax.set_ylabel('Total synaptic weight')
ax.set_title('FF / FB / Lateral weight balance')
for bar, p in zip(bars, pcts):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.2,
            f'{p:.1f}%', ha='center', fontsize=9)
plt.tight_layout()
plt.show()

## 4. Hub Detection — Rich-Club Inter-Neurons

**Hubs**: inter-neurons in the top quintile of total degree among inter-neurons.  
In the real FlyWire connectome ~30% of all neurons qualify as rich-club members (Lin et al. 2024).

**Controller prior**: Hub neurons aggregate many inputs and broadcast to many outputs.  
In a GNN controller they should carry the highest-dimensional hidden states —  
**layer-specific width allocation** (hub nodes → wider, peripheral nodes → narrower)  
is a direct structural inductive bias absent from uniform-width architectures like MLPs.

Hubs are also natural candidates for **hierarchical sub-policies**: because they bridge  
multiple sensorimotor channels, freezing their weights and fine-tuning only peripheral nodes  
is a principled transfer-learning strategy.

In [ ]:
inter_nodes    = [n for n in G.nodes if G.nodes[n]['layer'] == 'inter']
inter_total_deg = {n: in_deg[n] + out_deg[n] for n in inter_nodes}
threshold = np.percentile(list(inter_total_deg.values()), 80)
hubs = {n for n, d in inter_total_deg.items() if d >= threshold}

print(f"Hub threshold (80th percentile total degree, inter only): {threshold:.1f}")
print(f"Hub inter-neurons ({len(hubs)}): {sorted(hubs)}")

node_sizes  = [600 if n in hubs else 200 for n in G.nodes]
edge_widths = [3.0 if (u in hubs or v in hubs) else 0.6
               for u, v in G.edges]

fig, ax = plt.subplots(figsize=(11, 7))
nx.draw_networkx_nodes(G, pos, nodelist=node_list, node_color=node_colors,
                       node_size=node_sizes, ax=ax, alpha=0.9)
nx.draw_networkx_edges(G, pos, ax=ax, arrowsize=10, alpha=0.35,
                       width=edge_widths, edge_color='#444')
nx.draw_networkx_labels(G, pos, ax=ax, font_size=7)
ax.legend(handles=layer_patches, loc='upper left')
ax.set_title('Hub inter-neurons (large nodes = hub; thick edges = hub-incident)')
ax.axis('off')
plt.tight_layout()
plt.show()

## 5. Global Reaching Centrality — Hierarchy Score

**Global Reaching Centrality (GRC)** (Mones et al. 2012) measures how hierarchically organised  
a network is: GRC = 1 → perfect DAG (pure feedforward), GRC = 0 → no hierarchy.

**Controller prior**: Higher GRC → faster gradient propagation during early training (clear signal path).  
Lower GRC → richer temporal dynamics but harder to train from scratch.  
The real Drosophila connectome is intermediate — supporting both rapid reflexes and sustained integration.

In [ ]:
grc = nx.global_reaching_centrality(G, weight='weight', normalized=True)

ff_frac  = stats['feedforward'] / total
fb_frac  = stats['feedback']    / total
lat_frac = stats['lateral']     / total

print(f"Global Reaching Centrality (GRC) : {grc:.4f}  [0 = flat, 1 = pure DAG]")
print(f"Feedforward fraction              : {ff_frac:.3f}")
print(f"Feedback fraction                 : {fb_frac:.3f}")
print(f"Lateral fraction                  : {lat_frac:.3f}")

## 6. Summary: Controller Prior Interpretation

| Metric | Value (toy) | Controller implication |
|--------|-------------|------------------------|
| Global Reaching Centrality | see above | Hierarchy → reflex speed vs temporal integration trade-off |
| Feedforward fraction | see above | Reactive sensorimotor throughput |
| Feedback fraction | see above | Online correction; implement as residual skip-connections |
| Lateral fraction | see above | Working-memory capacity; keep recurrent edges in adj. matrix |
| Top-betweenness inter-neurons | ~3–5 nodes | Sub-policy hubs: widen hidden dim here |
| Hub fraction (top-quintile degree) | ~20% inter | Rich-club: allocate higher embedding dim to hub nodes |

### From analysis to GNN architecture (Project B)

1. **Node roles** map directly to GNN I/O: sensor nodes → input features; motor nodes → output heads.
2. **Message-passing depth** ≈ min path length `sensor → inter → motor` (2 hops baseline).
3. **Hub nodes** → assign wider MLP readout or higher embedding dimension.
4. **Feedback edges** → implement as residual connections or learnable skip-gates.
5. **Lateral recurrent edges** → keep in adjacency; create implicit memory without extra LSTM cells.
6. **Community structure** in inter-layer → candidate modular sub-policies (freeze module, fine-tune).

### Next steps — scaling to real FlyWire data

- Replace toy CSV with FlyWire v630 snapshot: [Zenodo:10676866](https://zenodo.org/records/10676866)
- Annotate nodes with `cell_type` from Schlegel et al. 2024 (sensorimotor, visual, olfactory…)
- Run this pipeline on the sensorimotor subgraph (~few thousand neurons)
- Use GRC + FF/FB/lateral + community partition as the structural specification for Project B controller

In [ ]:
summary = nodes_df.groupby('layer').agg(
    count         = ('node',         'count'),
    mean_in_deg   = ('in_degree',    'mean'),
    mean_out_deg  = ('out_degree',   'mean'),
    mean_bc       = ('betweenness',  'mean'),
    max_bc        = ('betweenness',  'max'),
).round(3)

print("Per-layer structural summary:")
print(summary.to_string())

print(f"\nNetwork-level metrics:")
print(f"  Nodes              : {G.number_of_nodes()}")
print(f"  Edges              : {G.number_of_edges()}")
print(f"  Density            : {nx.density(G):.4f}")
print(f"  Global GRC         : {grc:.4f}")
print(f"  Hub inter-neurons  : {sorted(hubs)}")